In [17]:
import os
import torch
from collections import defaultdict
import logging
import pickle
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset


/home/mrenaudin/.conda/envs/leaps3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 352462.52 examples/s]


In [26]:
def create_vocab(train_data, vocab_size):
    counter = defaultdict(int)
    for line in train_data:
        for word in line.replace("\n"," <eos>").split():
            counter[word] += 1

    count_pairs = sorted(counter.items(), key=lambda x: (-x[1], x[0]))[:vocab_size]
    words = [w for (w, v) in count_pairs]
    print(len(counter), count_pairs[vocab_size - 1])
    w2idx = dict(zip(words, range(len(words))))
    idx2w = dict(zip(range(len(words)), words))
    return w2idx, idx2w

In [37]:
w2i, i2w = create_vocab(ds["train"]["text"], 50000)

623942 ('Masand', 62)


In [38]:
len(w2i)

50000

In [39]:
def filter_word(word, vocab):
    if word in vocab:
        return word
    else:
        return "<unk>"

In [40]:
def convert_line(line, vocab):
    return [filter_word(word, vocab) for word in line.replace("\n", " <eos>").split()]

In [41]:
output_path = "/scratch2/mrenaudin/colorlessgreenRNNs/new_dataset"

In [42]:
f_train = open(output_path + "/train.txt", 'w')


In [43]:
for line in ds["train"]["text"]:
    f_train.write(" ".join(convert_line(line, w2i)) + "\n")
f_train.close()

Meme implementation qu'avant, juste avec un dataset et un dataloader.
On garde indice pour les phrase et on shuffle ça dans le dataloader

In [44]:
import os
import torch
from collections import defaultdict
import logging

In [45]:
class Dictionary(object):
    def __init__(self, path):
        self.word2idx = {}
        self.idx2word = []
        self.word2freq = defaultdict(int)

        vocab_path = os.path.join(path, 'vocab.txt')
        try:
            vocab = open(vocab_path, encoding="utf8").read()
            self.word2idx = {w: i for i, w in enumerate(vocab.split())}
            self.idx2word = [w for w in vocab.split()]
            self.vocab_file_exists = True
        except FileNotFoundError:
            logging.info("Vocab file not found, creating new vocab file.")
            self.create_vocab(os.path.join(path, 'train.txt'))
            open(vocab_path,"w").write("\n".join([w for w in self.idx2word]))

    def add_word(self, word):
        self.word2freq[word] += 1
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
        #return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)

    def create_vocab(self, path):
        with open(path, 'r', encoding="utf8") as f:
            for line in f:
                words = line.split()
                for word in words:
                    self.add_word(word)

In [55]:
class Corpus(object):
    def __init__(self, path):
        self.dictionary = Dictionary(path)
        #self.train = tokenize(self.dictionary, os.path.join(path, 'train.txt'))
        self.valid = tokenize(self.dictionary, os.path.join(path, 'valid.txt'), shuffle=False)
        self.test = tokenize(self.dictionary, os.path.join(path, 'test.txt'), shuffle=False)


In [ ]:
corpus.train = tokenize(self.dictionary, os.path.join(path, 'train.txt'), shuffle=True)

In [47]:
import random
def tokenize(dictionary, path, shuffle=False):
    """Tokenizes a text file for training or testing to a sequence of indices format
       We assume that training and test data has <eos> symbols """
    assert os.path.exists(path)
    with open(path, 'r', encoding="utf8") as f:
        lines = f.readlines()

    if shuffle:
        random.shuffle(lines)
        ntokens = 0
        for line in lines:
            words = line.split()
            ntokens += len(words)

    # Tokenize file content
    with open(path, 'r', encoding="utf8") as f:
        ids = torch.LongTensor(ntokens)
        token = 0
        for line in lines:
            words = line.split()
            for word in words:
                if word in dictionary.word2idx:
                    ids[token] = dictionary.word2idx[word]
                else:
                    ids[token] = dictionary.word2idx["<unk>"]
                token += 1

    return ids

In [57]:
import os
import torch
import random

def tokenize(dictionary, path, shuffle=False):
    """Tokenizes a text file to a sequence of indices format.
       Assumes that training and test data have <eos> symbols.
    """
    assert os.path.exists(path)

    # Read all lines
    with open(path, 'r', encoding="utf8") as f:
        lines = f.readlines()

    if shuffle:
        random.shuffle(lines)

    # Count total number of tokens
    ntokens = 0
    for line in lines:
        words = line.split()
        ntokens += len(words)

    # Allocate tensor
    ids = torch.LongTensor(ntokens)

    # Fill tensor
    token = 0
    for line in lines:
        words = line.split()
        for word in words:
            if word in dictionary.word2idx:
                ids[token] = dictionary.word2idx[word]
            else:
                ids[token] = dictionary.word2idx.get("<unk>", 0)
            token += 1

    return ids


In [49]:
def batchify(data, bsz, device):
    #Just add shuffling ici
    # Work out how cleanly we can divide the dataset into bsz parts.
    nbatch = data.size(0) // bsz
    # Trim off any extra elements that wouldn't cleanly fit (remainders).
    data = data.narrow(0, 0, nbatch * bsz)
    # Evenly divide the data across the bsz batches.
    data = data.view(bsz, -1).t().contiguous()
    # if device = 'cuda':
    #     #data = data.cuda()
    data = data.to(device)
    return data

In [50]:
def get_batch(source, i, seq_length):
    """Gets a single batch from source data at position i"""
    seq_len = min(seq_length, len(source) - 1 - i)
    data = source[i : i + seq_len]
    # predict the sequences shifted by one word
    target = source[i + 1 : i + 1 + seq_len].view(-1)
    return data, target

In [51]:
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')

In [53]:
print(corpus.train.shape)
print(corpus.valid.shape)
print(corpus.test.shape)


torch.Size([83058298])
torch.Size([10391172])
torch.Size([10366477])


In [58]:
corpus2 = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')

In [59]:
corpus2.train = tokenize(corpus2.dictionary, os.path.join('/scratch2/mrenaudin/colorlessgreenRNNs/english_data', 'train.txt'), shuffle=True)

In [64]:
corpus2.train.shape
corpus2.valid.shape
corpus2.test.shape

torch.Size([10366477])